# Carnet 2 : Évaluation Avancée et Registre de Modèle

Jusqu'à présent, j'ai simplement surveillé des métriques basiques. Mais je me rends compte que pour vraiment comprendre le comportement de mes modèles (surtout pour prédire la gravité d'un accident), j'ai besoin d'aller plus loin.

L'idée de ce carnet est de tester différentes configurations pour Gradient Boosting, puis XGBoost, mais en générant des artefacts visuels (matrices de confusion, courbes ROC, feature importance). Ensuite, j'aimerais voir s'il est possible d'isoler automatiquement le meilleur modèle pour le sauvegarder dans un "Model Registry" et l'utiliser pour faire des prédictions. On va voir ce que ça donne !

## 1. Imports, Configuration et Chargement des données

In [1]:
from datetime import datetime
import tempfile
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, log_loss, ConfusionMatrixDisplay, roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.ensemble import GradientBoostingClassifier
import xgboost as xgb

import mlflow
from mlflow.tracking import MlflowClient
import logging
logging.getLogger("mlflow.sklearn").setLevel(logging.ERROR) 

# Je configure le tracking MLflow avec une base SQLite locale
mlflow.set_tracking_uri("http://localhost:5000")

# Chargement de mon dataset
df_accident = pd.read_csv('../data/dataset_accident.csv', sep=';' )

# Séparation de la cible et des features pour éviter toute fuite de données
y = df_accident["grav_binary"]
X = df_accident.drop(columns=["grav_ordered", "grav_binary"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 2. Baseline XGBoost

Le Gradient Boosting c'est bien, mais j'ai entendu dire que XGBoost était plus rapide, qu'il gérait nativement les valeurs manquantes et qu'il avait une régularisation intégrée qui limite le surapprentissage. 

Je vais d'abord construire une première ligne de base (baseline) avec XGBoost.

In [2]:
experiment_name = "ManualTuning"
mlflow.set_experiment(experiment_name)

with mlflow.start_run(run_name=f"XGBoost_baseline {datetime.now().strftime('%d/%m/%Y %Hh:%Mm:%Ss')}"):
    
    model = xgb.XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        eval_metric="logloss",
        random_state=42
    )
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)

    mlflow.log_params(model.get_params())
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred, average='weighted'),
        "precision": precision_score(y_test, y_pred, average='weighted'),
        "recall": recall_score(y_test, y_pred, average='weighted'),
        "log_loss": log_loss(y_test, y_pred_proba)
    }
    mlflow.log_metrics(metrics)
    mlflow.xgboost.log_model(model, name="XGBoost_baseline")

    with tempfile.TemporaryDirectory() as tmpdir:
        fig, ax = plt.subplots(figsize=(8, 6))
        ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax)
        ax.set_title("Confusion Matrix - XGBoost_baseline")
        plt.tight_layout()
        cm_path = os.path.join(tmpdir, "confusion_matrix.png")
        plt.savefig(cm_path)
        mlflow.log_artifact(cm_path)
        plt.close()

        fig, ax = plt.subplots(figsize=(8, 6))
        fpr, tpr, _ = roc_curve(y_test, y_pred_proba[:, 1])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
        ax.plot([0, 1], [0, 1], 'k--')
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.set_title("Courbes ROC - XGBoost_baseline")
        ax.legend()
        plt.tight_layout()
        roc_path = os.path.join(tmpdir, "roc_curves.png")
        plt.savefig(roc_path)
        mlflow.log_artifact(roc_path)
        plt.close()

2026/02/26 17:28:58 INFO mlflow.tracking.fluent: Experiment with name 'ManualTuning' does not exist. Creating a new experiment.


🏃 View run XGBoost_baseline 26/02/2026 17h:28m:58s at: http://localhost:5000/#/experiments/2/runs/9a17b83c91974888acf8ecf2127973bb
🧪 View experiment at: http://localhost:5000/#/experiments/2


Maintenant, j'aimerais essayer 3 variantes manuelles pour ce XGBoost. On va voir s'il est possible d'améliorer légèrement nos résultats en tatonnant un peu avec les paramètres.

In [3]:
configs_xgb = [
    {"name": "XGB_simple", "n_estimators": 50, "learning_rate": 0.3, "max_depth": 3},
    {"name": "XGB_balanced", "n_estimators": 150, "learning_rate": 0.1, "max_depth": 6},
    {"name": "XGB_agressive", "n_estimators": 500, "learning_rate": 0.01, "max_depth": 9}
]

for config in configs_xgb:
    current_config = config.copy()
    run_name = current_config.pop("name")
    
    with mlflow.start_run(run_name=f"{run_name} {datetime.now().strftime('%d/%m/%Y %Hh:%Mm:%Ss')}"):
        model = xgb.XGBClassifier(
            **current_config,
            eval_metric="logloss",
            random_state=42
        )
        
        model.fit(X_train, y_train)
        
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)
        
        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "f1_score": f1_score(y_test, y_pred, average='weighted'),
            "precision": precision_score(y_test, y_pred, average='weighted'),
            "recall": recall_score(y_test, y_pred, average='weighted'),
            "log_loss": log_loss(y_test, y_pred_proba)
        }
        
        mlflow.log_params(model.get_params())
        mlflow.log_metrics(metrics)
        mlflow.xgboost.log_model(model, name=run_name)

        with tempfile.TemporaryDirectory() as tmpdir:
            fig, ax = plt.subplots(figsize=(8, 6))
            ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax)
            ax.set_title(f"Confusion Matrix - {run_name}")
            plt.tight_layout()
            cm_path = os.path.join(tmpdir, "confusion_matrix.png")
            plt.savefig(cm_path)
            mlflow.log_artifact(cm_path)
            plt.close()
        
        print(f"Run {run_name} terminé - Recall: {metrics['recall']:.4f}")

Run XGB_simple terminé - Recall: 0.7202
🏃 View run XGB_simple 26/02/2026 17h:29m:10s at: http://localhost:5000/#/experiments/2/runs/323fd0d717ba420189f862db19730a56
🧪 View experiment at: http://localhost:5000/#/experiments/2
Run XGB_balanced terminé - Recall: 0.7198
🏃 View run XGB_balanced 26/02/2026 17h:29m:20s at: http://localhost:5000/#/experiments/2/runs/bc2c190e9ead460d904a109b775fbc25
🧪 View experiment at: http://localhost:5000/#/experiments/2
Run XGB_agressive terminé - Recall: 0.7201
🏃 View run XGB_agressive 26/02/2026 17h:29m:32s at: http://localhost:5000/#/experiments/2/runs/48cea146d33041f59c9aec37465c2b91
🧪 View experiment at: http://localhost:5000/#/experiments/2


## 3. Extraction automatique du meilleur modèle

Plutôt que d'aller scruter l'interface MLflow et de retenir manuellement le meilleur modèle, je me demande si je peux le faire directement par le code. L'API `MlflowClient` semble parfaite pour ça.

Mon objectif principal, c'est de ne surtout pas rater d'accidents graves (les Faux Négatifs sont dangereux). Je vais donc cibler le modèle avec le meilleur Recall.

In [4]:
client = MlflowClient()
experiment = client.get_experiment_by_name(experiment_name)
experiment_id = experiment.experiment_id

# Je filtre les runs en les triant par la métrique 'recall' de façon décroissante
runs = client.search_runs(
    experiment_ids=[experiment_id],
    order_by=["metrics.recall DESC"] 
)

if runs:
    best_run = runs[0]
    best_run_id = best_run.info.run_id
    print(f"Le modèle retenu est : {best_run.info.run_name}")
    print(f"Son Recall atteint : {best_run.data.metrics['recall']:.4f}")
else:
    print("Mince, aucun run trouvé...")

Le modèle retenu est : XGBoost_baseline 26/02/2026 17h:28m:58s
Son Recall atteint : 0.7209


## 4. Mise en registre et simulation de déploiement

J'ai localisé le modèle. Maintenant, je veux le sauvegarder officiellement dans un espace central : le Model Registry de MLflow. L'idée est de pouvoir l'utiliser plus tard juste en l'appelant par son nom, sans avoir besoin du script d'entraînement.

In [5]:
model_name_registry = "Detecteur_Accidents_Route"

if runs:
    # MLflow 3.x : les modèles ne sont plus dans les artefacts classiques
    logged_models = mlflow.search_logged_models(
        experiment_ids=[experiment_id],
        filter_string=f"source_run_id = '{best_run_id}'"
    )
    
    if not logged_models.empty:
        model_id = logged_models.iloc[0]["model_id"]
        model_uri = f"models:/{model_id}"
        result = mlflow.register_model(model_uri, model_name_registry)
        print(f"Génial, le modèle est enregistré sous le nom : {model_name_registry}")
    else:
        print("Aucun modèle loggé trouvé pour ce run.")
else:
    print("Impossible d'enregistrer, je n'ai pas de run de référence.")

Successfully registered model 'Detecteur_Accidents_Route'.
2026/02/26 17:29:50 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Detecteur_Accidents_Route, version 1


Génial, le modèle est enregistré sous le nom : Detecteur_Accidents_Route


Created version '1' of model 'Detecteur_Accidents_Route'.


Imaginons maintenant qu'on soit dans une application de secours en direct. 
Je vais essayer de charger ce modèle directement depuis le registre et de lui passer de nouvelles données (que je génère au hasard pour simuler une situation réelle) pour voir ce qu'il me sort comme prédictions.

In [6]:
try:
    # Je charge la toute dernière version de mon modèle enregistré
    model_deploy = mlflow.xgboost.load_model(f"models:/{model_name_registry}/latest")
    print("Le modèle du registre a bien été chargé !\n")

    # Je génère 10 fausses situations d'accidents pour tester le modèle
    np.random.seed(42)
    n_rows = 10

    data = {
        'est_nuit': np.random.randint(0, 2, n_rows),
        'est_heure_pointe': np.random.randint(0, 2, n_rows),
        'jour_semaine': np.random.randint(1, 8, n_rows),
        'est_weekend': np.random.randint(0, 2, n_rows),
        'agg': np.random.randint(0, 2, n_rows),
        'vma': np.random.choice([30, 50, 70, 80, 110, 130], n_rows),
        'impl_vehicule_leger': np.random.randint(0, 2, n_rows),
        'impl_poids_lourd': np.random.randint(0, 2, n_rows),
        'impl_pieton': np.random.randint(0, 2, n_rows),
    }

    df_simul = pd.DataFrame(data)

    # Je simule arbitrairement ce qui défini un cas grave dans ma fausse réalité
    score = (df_simul['est_nuit'] * 2) + (df_simul['vma'] / 50) + (df_simul['impl_poids_lourd'] * 3) + (df_simul['impl_pieton'] * 4)
    df_simul['grav_binary'] = (score > 6).astype(int)
    df_simul = df_simul[['grav_binary'] + [c for c in df_simul.columns if c != 'grav_binary']]

    X_new = df_simul.drop(columns=['grav_binary'])
    y_true = df_simul['grav_binary']

    # Je demande au modèle ce qu'il en pense
    y_pred = model_deploy.predict(X_new)
    df_simul['predictions'] = y_pred

    print("Comparatif entre la réalité simulée et ce que le modèle prédit :")
    print(df_simul[['grav_binary', 'predictions']])

    print("\n--- Petit aperçu des perfs sur ces nouvelles données ---")
    print(classification_report(y_true, y_pred, zero_division=0))
    
    rec = recall_score(y_true, y_pred, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    print(f"Précision globale sur ces {n_rows} tests : {acc:.2%}")
    print(f"Taux de détection des cas graves (Recall) : {rec:.2%}")
except Exception as e:
    print(f"Oups, un problème est survenu : {e}")

Le modèle du registre a bien été chargé !

Comparatif entre la réalité simulée et ce que le modèle prédit :
   grav_binary  predictions
0            1            1
1            1            1
2            0            1
3            1            1
4            0            1
5            1            1
6            0            1
7            1            1
8            0            1
9            1            1

--- Petit aperçu des perfs sur ces nouvelles données ---
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         4
           1       0.60      1.00      0.75         6

    accuracy                           0.60        10
   macro avg       0.30      0.50      0.38        10
weighted avg       0.36      0.60      0.45        10

Précision globale sur ces 10 tests : 60.00%
Taux de détection des cas graves (Recall) : 100.00%


## Conclusion : Les limites de l'approche manuelle

Même si ce carnet a permis de suivre tout le cycle de vie d'un modèle (de l'entraînement jusqu'à un registre), je me rends compte que la façon de chercher les hyperparamètres "à la main" n'est vraiment pas optimale.

J'ai pu tester 3 ou 4 configurations un peu au hasard, mais cela prend beaucoup de temps, c'est très subjectif et on risque de passer à côté du vrai potentiel de notre XGBoost. Les paramètres interagissent entre eux : on ne peut pas les deviner facilement sans une bonne approche mathématique.

Dans le **prochain carnet (Carnet 3)**, je vais essayer d'automatiser cette recherche d'hyperparamètres ! On verra comment s'appuyer sur des bibliothèques réputées comme **GridSearchCV**, **Hyperopt**, et **Optuna** pour laisser l'ordinateur trouver la configuration parfaite lui-même.


[-- Ouvrir le Carnet 3 : Tuning Avancé --](03_Tuning_Avance_Auto.ipynb)